# Análise dos Dados

## Objetivo deste notebook

Este notebook apresenta a etapa final de análise do MVP de Engenharia de Dados.

As análises serão realizadas a partir das tabelas da camada Gold, construídas após as etapas de ingestão, tratamento, padronização, verificação da qualidade e modelagem dos dados.

O objetivo é utilizar os dados preparados para responder às quatro perguntas de negócio definidas no início do projeto:

1. Como a carga de energia elétrica varia ao longo do tempo nos diferentes subsistemas do Sistema Interligado Nacional?

2. Como está distribuída a geração de energia elétrica entre as diferentes fontes de geração, como hidrelétrica, térmica, eólica e solar?

3. Como evoluiu a participação das diferentes fontes de geração de energia elétrica ao longo do período analisado?

4. Como o comportamento da geração de energia elétrica se relaciona com as variações da carga nos diferentes subsistemas do SIN?

As análises considerarão o período de **2020 a 2025** e utilizarão indicadores mensais preparados na camada Gold.


### Carregamento dos dados de carga

Inicialmente, é carregada a tabela `carga_mensal_subsistema` da camada Gold. Essa tabela contém os indicadores mensais de carga preparados nas etapas anteriores do pipeline.

Após o carregamento, são verificadas a quantidade de registros e de colunas da tabela, além da exibição de uma amostra dos dados. Essa verificação permite confirmar que a estrutura analítica está disponível corretamente antes do início das análises.

In [0]:
tabela_gold_carga = "workspace.energia_gold.carga_mensal_subsistema"

df_carga = spark.table(tabela_gold_carga)

print("Quantidade de registros:", df_carga.count())
print("Quantidade de colunas:", len(df_carga.columns))

display(df_carga.limit(20))

Quantidade de registros: 360
Quantidade de colunas: 7


ano,mes,id_subsistema,nom_subsistema,carga_media,carga_minima,carga_maxima
2020,1,N,NORTE,5412.0,4260.09,6101.37
2020,1,NE,NORDESTE,10952.56,8203.38,12787.02
2020,1,S,SUL,12599.13,7791.72,17403.36
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,27453.61,51230.32
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,49362.19,84879.02
2020,2,N,NORTE,5513.53,4523.39,6355.08
2020,2,NE,NORDESTE,10995.71,8853.08,12983.98
2020,2,S,SUL,12781.99,7703.28,18002.31
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,27405.48,50626.06
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,49256.81,86509.74


## Pergunta 1 — Evolução da carga de energia elétrica

**Como a carga de energia elétrica varia ao longo do tempo nos diferentes subsistemas do Sistema Interligado Nacional?**

Para responder a essa pergunta, será analisada a evolução mensal da carga média entre 2020 e 2025.

O registro referente ao SIN representa o Sistema Interligado Nacional de forma agregada e será mantido separado dos quatro subsistemas regionais nas visualizações, evitando comparar diretamente níveis de agregação diferentes.

### Preparação dos dados para análise temporal

Para facilitar a análise da evolução da carga ao longo do tempo, os campos `ano` e `mes` são utilizados para criar a coluna `data`, representando o primeiro dia de cada mês.

Em seguida, são selecionados os atributos necessários para a análise — identificação do subsistema e indicadores de carga média, mínima e máxima — e os registros são ordenados cronologicamente.

Essa preparação permite utilizar uma dimensão temporal adequada nas consultas e visualizações realizadas nas etapas seguintes.

In [0]:
from pyspark.sql.functions import make_date,lit

df_carga_analise = (
    df_carga
    .withColumn(
        "data",
        make_date("ano", "mes", lit(1))
    )
    .select(
        "data",
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema",
        "carga_media",
        "carga_minima",
        "carga_maxima"
    )
    .orderBy("data", "id_subsistema")
)

display(df_carga_analise.limit(20))

data,ano,mes,id_subsistema,nom_subsistema,carga_media,carga_minima,carga_maxima
2020-01-01,2020,1,N,NORTE,5412.0,4260.09,6101.37
2020-01-01,2020,1,NE,NORDESTE,10952.56,8203.38,12787.02
2020-01-01,2020,1,S,SUL,12599.13,7791.72,17403.36
2020-01-01,2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,27453.61,51230.32
2020-01-01,2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,49362.19,84879.02
2020-02-01,2020,2,N,NORTE,5513.53,4523.39,6355.08
2020-02-01,2020,2,NE,NORDESTE,10995.71,8853.08,12983.98
2020-02-01,2020,2,S,SUL,12781.99,7703.28,18002.31
2020-02-01,2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,27405.48,50626.06
2020-02-01,2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,49256.81,86509.74


### Evolução mensal da carga por subsistema

Para comparar a evolução da carga entre os subsistemas regionais, o registro agregado do SIN é retirado desta visualização, mantendo apenas Norte, Nordeste, Sul e Sudeste/Centro-Oeste.

São selecionadas a data, a identificação do subsistema e a carga média mensal. Os registros são ordenados cronologicamente e utilizados na construção de um gráfico de linhas, permitindo visualizar e comparar o comportamento da carga nos quatro subsistemas ao longo do período de 2020 a 2025.

In [0]:
df_carga_subsistemas = (
    df_carga_analise
    .filter("id_subsistema <> 'SIN'")
    .select(
        "data",
        "nom_subsistema",
        "carga_media"
    )
    .orderBy("data", "nom_subsistema")
)

display(df_carga_subsistemas)

data,nom_subsistema,carga_media
2020-01-01,NORDESTE,10952.56
2020-01-01,NORTE,5412.0
2020-01-01,SUDESTE/CENTRO-OESTE,39577.57
2020-01-01,SUL,12599.13
2020-02-01,NORDESTE,10995.71
2020-02-01,NORTE,5513.53
2020-02-01,SUDESTE/CENTRO-OESTE,39505.45
2020-02-01,SUL,12781.99
2020-03-01,NORDESTE,10597.1
2020-03-01,NORTE,5347.47


Databricks visualization. Run in Databricks to view.

### Comparação da carga média entre os subsistemas

Além da evolução mensal, é calculada a carga média de todo o período analisado para cada subsistema regional.

Os registros são agrupados por subsistema e a média da carga é calculada considerando os dados mensais de 2020 a 2025. Os resultados são arredondados para duas casas decimais e ordenados do maior para o menor valor.

Essa comparação permite identificar as diferenças no nível médio de carga entre os quatro subsistemas ao longo do período analisado.

In [0]:
from pyspark.sql.functions import avg, round

df_resumo_carga = (
    df_carga_subsistemas
    .groupBy("nom_subsistema")
    .agg(
        round(
            avg("carga_media"), 2
        ).alias("carga_media_periodo")
    )
    .orderBy("carga_media_periodo", ascending=False)
)

display(df_resumo_carga)

nom_subsistema,carga_media_periodo
SUDESTE/CENTRO-OESTE,40967.82
SUL,12491.5
NORDESTE,11896.91
NORTE,6829.59


### Comparação da carga entre 2020 e 2025

Para avaliar a mudança da carga entre o início e o final do período analisado, são selecionados os anos de 2020 e 2025 para os quatro subsistemas regionais.

Os dados são agrupados por ano e subsistema, sendo calculada a carga média anual a partir dos valores mensais disponíveis na camada Gold.

Essa comparação permite observar como a carga média de cada subsistema se modificou entre 2020 e 2025 e servirá de base para o cálculo da variação percentual realizado na etapa seguinte.

In [0]:
from pyspark.sql.functions import avg, round

df_comparacao_carga = (
    df_carga_analise
    .filter(
        (df_carga_analise.id_subsistema != "SIN") &
        (df_carga_analise.ano.isin(2020, 2025))
    )
    .groupBy("ano", "nom_subsistema")
    .agg(
        round(avg("carga_media"), 2).alias("carga_media_anual")
    )
    .orderBy("nom_subsistema", "ano")
)

display(df_comparacao_carga)

ano,nom_subsistema,carga_media_anual
2020,NORDESTE,10300.69
2025,NORDESTE,13271.69
2020,NORTE,5516.36
2025,NORTE,8309.1
2020,SUDESTE/CENTRO-OESTE,36310.93
2025,SUDESTE/CENTRO-OESTE,44268.22
2020,SUL,11289.38
2025,SUL,13818.95


### Variação percentual da carga entre 2020 e 2025

A partir das cargas médias anuais calculadas anteriormente, os valores de 2020 e 2025 são reorganizados em colunas separadas para permitir uma comparação direta entre o início e o final do período.

Em seguida, é calculada a variação percentual da carga de cada subsistema entre os dois anos. O resultado é arredondado para duas casas decimais e ordenado de forma decrescente pela variação percentual.

Esse cálculo permite identificar proporcionalmente quanto a carga média aumentou ou diminuiu em cada subsistema entre 2020 e 2025.

In [0]:
from pyspark.sql.functions import col, round

df_variacao_carga = (
    df_comparacao_carga
    .groupBy("nom_subsistema")
    .pivot("ano", [2020, 2025])
    .agg(avg("carga_media_anual"))
    .withColumnRenamed("2020", "carga_2020")
    .withColumnRenamed("2025", "carga_2025")
    .withColumn(
        "variacao_percentual",
        round(
            ((col("carga_2025") - col("carga_2020")) /
             col("carga_2020")) * 100,
            2
        )
    )
    .orderBy("variacao_percentual", ascending=False)
)

display(df_variacao_carga)

nom_subsistema,carga_2020,carga_2025,variacao_percentual
NORTE,5516.36,8309.1,50.63
NORDESTE,10300.69,13271.69,28.84
SUL,11289.38,13818.95,22.41
SUDESTE/CENTRO-OESTE,36310.93,44268.22,21.91


### Análise da Pergunta 1

A análise da carga média mensal entre 2020 e 2025 demonstra diferenças relevantes entre os quatro subsistemas do Sistema Interligado Nacional.

O subsistema **Sudeste/Centro-Oeste** apresentou o maior nível médio de carga durante o período analisado, com média de aproximadamente **40.967,82**, seguido pelos subsistemas **Sul (12.491,50)**, **Nordeste (11.896,91)** e **Norte (6.829,59)**.

Além das diferenças de magnitude entre os subsistemas, o gráfico de evolução mensal evidencia oscilações da carga ao longo do período. Apesar dessas variações mensais, a comparação das médias anuais de 2020 e 2025 mostra crescimento nos quatro subsistemas.

As variações observadas foram:

- **Norte:** crescimento de **50,63%**;
- **Nordeste:** crescimento de **28,84%**;
- **Sul:** crescimento de **22,41%**;
- **Sudeste/Centro-Oeste:** crescimento de **21,91%**.

Portanto, os resultados indicam uma **tendência de crescimento da carga ao longo do período analisado**, embora com intensidades diferentes entre os subsistemas. O Sudeste/Centro-Oeste permanece com o maior nível de carga, enquanto o Norte apresentou o maior crescimento percentual na comparação entre as médias anuais de 2020 e 2025.

Esses resultados são descritivos e demonstram o comportamento observado nos dados analisados, não permitindo, isoladamente, determinar as causas das variações identificadas.


## Pergunta 2 — Distribuição da geração por fonte

**Como está distribuída a geração de energia elétrica entre as diferentes fontes de geração, como hidrelétrica, térmica, eólica e solar?**

Para responder a essa pergunta, serão analisadas as fontes de geração hidráulica, térmica, eólica e solar disponíveis no conjunto de dados.

Nesta análise, será utilizada inicialmente a informação correspondente ao **Sistema Interligado Nacional (SIN)**, permitindo observar a distribuição das quatro fontes em nível agregado sem somar novamente os valores dos subsistemas regionais.

### Carregamento dos dados de geração

Para iniciar a análise da distribuição das fontes de geração, é carregada a tabela `geracao_mensal_subsistema` da camada Gold.

Essa tabela contém os indicadores mensais de geração hidráulica, térmica, eólica e solar para os diferentes identificadores do conjunto de dados no período de 2020 a 2025.

Após o carregamento, são verificadas a quantidade de registros e de colunas da tabela e exibida uma amostra dos dados. Essa etapa permite confirmar que a estrutura analítica necessária para responder à segunda pergunta de negócio está disponível corretamente.

In [0]:
tabela_gold_geracao = "workspace.energia_gold.geracao_mensal_subsistema"

df_geracao = spark.table(tabela_gold_geracao)

print("Quantidade de registros:", df_geracao.count())
print("Quantidade de colunas:", len(df_geracao.columns))

display(df_geracao.limit(20))

Quantidade de registros: 360
Quantidade de colunas: 8


ano,mes,id_subsistema,nom_subsistema,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media
2020,1,N,NORTE,8188.34,2861.9,126.65,0.0
2020,1,NE,NORDESTE,2897.49,2322.59,2965.7,317.59
2020,1,S,SUL,4523.55,1817.24,517.36,0.0
2020,1,SE,SUDESTE/CENTRO-OESTE,35469.7,6309.17,0.0,192.27
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,51079.08,13310.89,3609.71,509.87
2020,2,N,NORTE,12613.98,2140.81,99.05,0.0
2020,2,NE,NORDESTE,2883.37,1697.99,3086.87,333.01
2020,2,S,SUL,3361.24,1087.89,708.2,0.0
2020,2,SE,SUDESTE/CENTRO-OESTE,36109.29,4494.53,0.0,180.45
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,54967.88,9421.22,3894.12,513.46


### Seleção dos dados agregados do SIN

Para analisar a distribuição das fontes de geração em nível agregado, são selecionados apenas os registros identificados como `SIN`, correspondentes ao Sistema Interligado Nacional.

Essa seleção evita somar novamente os valores dos subsistemas regionais, pois o próprio conjunto de dados já disponibiliza registros específicos para o SIN.

Os registros são ordenados por ano e mês e é realizada uma verificação da quantidade de observações disponíveis. Para o período de 2020 a 2025, são esperados 72 registros mensais, correspondentes a 6 anos × 12 meses.

In [0]:
df_geracao_sin = (
    df_geracao
    .filter("id_subsistema = 'SIN'")
    .orderBy("ano", "mes")
)

print("Quantidade de registros do SIN:", df_geracao_sin.count())

display(df_geracao_sin.limit(20))

Quantidade de registros do SIN: 72


ano,mes,id_subsistema,nom_subsistema,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,51079.08,13310.89,3609.71,509.87
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,54967.88,9421.22,3894.12,513.46
2020,3,SIN,SISTEMA INTERLIGADO NACIONAL,54467.28,8102.96,2977.47,563.82
2020,4,SIN,SISTEMA INTERLIGADO NACIONAL,44804.02,7564.67,4220.92,608.95
2020,5,SIN,SISTEMA INTERLIGADO NACIONAL,42303.18,8014.12,5059.82,594.18
2020,6,SIN,SISTEMA INTERLIGADO NACIONAL,41831.76,8384.58,6726.13,614.81
2020,7,SIN,SISTEMA INTERLIGADO NACIONAL,43943.62,7128.67,7679.74,635.58
2020,8,SIN,SISTEMA INTERLIGADO NACIONAL,43272.91,7813.71,9036.29,654.79
2020,9,SIN,SISTEMA INTERLIGADO NACIONAL,47328.25,8295.0,8975.17,696.05
2020,10,SIN,SISTEMA INTERLIGADO NACIONAL,44599.75,14141.78,7518.17,610.32


### Geração média por fonte no SIN

A partir dos registros mensais do Sistema Interligado Nacional, é calculada a geração média de cada uma das quatro fontes consideradas no projeto: hidráulica, térmica, eólica e solar.

O cálculo utiliza os 72 registros mensais do período de 2020 a 2025 e produz um valor médio para cada fonte, arredondado para duas casas decimais.

Esse resumo permite comparar a magnitude média das diferentes fontes de geração no SIN e servirá de base para o cálculo de suas participações percentuais.

In [0]:
from pyspark.sql.functions import avg, round

df_resumo_geracao_sin = (
    df_geracao_sin
    .agg(
        round(avg("geracao_hidraulica_media"), 2).alias("hidraulica"),
        round(avg("geracao_termica_media"), 2).alias("termica"),
        round(avg("geracao_eolica_media"), 2).alias("eolica"),
        round(avg("geracao_solar_media"), 2).alias("solar")
    )
)

display(df_resumo_geracao_sin)

hidraulica,termica,eolica,solar
47370.91,10554.89,9988.17,4424.23


### Carregamento dos dados de participação das fontes

Para calcular a distribuição percentual das fontes de geração, é carregada a tabela `participacao_fontes_mensal` da camada Gold.

Essa tabela contém, além dos valores médios de geração, a participação percentual mensal das fontes hidráulica, térmica, eólica e solar em relação ao total das quatro fontes consideradas no projeto.

Para esta análise são selecionados somente os registros correspondentes ao SIN, evitando a combinação de diferentes níveis de agregação. Os dados são ordenados por ano e mês e é verificada a disponibilidade dos 72 registros mensais do período de 2020 a 2025.

In [0]:
tabela_gold_participacao = "workspace.energia_gold.participacao_fontes_mensal"

df_participacao = spark.table(tabela_gold_participacao)

df_participacao_sin = (
    df_participacao
    .filter("id_subsistema = 'SIN'")
    .orderBy("ano", "mes")
)

print("Quantidade de registros do SIN:", df_participacao_sin.count())

display(df_participacao_sin.limit(20))

Quantidade de registros do SIN: 72


ano,mes,id_subsistema,nom_subsistema,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media,participacao_hidraulica_pct,participacao_termica_pct,participacao_eolica_pct,participacao_solar_pct
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,51079.08,13310.89,3609.71,509.87,68509.55,74.56,19.43,5.27,0.74
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,54967.88,9421.22,3894.12,513.46,68796.68000000001,79.9,13.69,5.66,0.75
2020,3,SIN,SISTEMA INTERLIGADO NACIONAL,54467.28,8102.96,2977.47,563.82,66111.53,82.39,12.26,4.5,0.85
2020,4,SIN,SISTEMA INTERLIGADO NACIONAL,44804.02,7564.67,4220.92,608.95,57198.55999999999,78.33,13.23,7.38,1.06
2020,5,SIN,SISTEMA INTERLIGADO NACIONAL,42303.18,8014.12,5059.82,594.18,55971.3,75.58,14.32,9.04,1.06
2020,6,SIN,SISTEMA INTERLIGADO NACIONAL,41831.76,8384.58,6726.13,614.81,57557.28,72.68,14.57,11.69,1.07
2020,7,SIN,SISTEMA INTERLIGADO NACIONAL,43943.62,7128.67,7679.74,635.58,59387.61,73.99,12.0,12.93,1.07
2020,8,SIN,SISTEMA INTERLIGADO NACIONAL,43272.91,7813.71,9036.29,654.79,60777.700000000004,71.2,12.86,14.87,1.08
2020,9,SIN,SISTEMA INTERLIGADO NACIONAL,47328.25,8295.0,8975.17,696.05,65294.47,72.48,12.7,13.75,1.07
2020,10,SIN,SISTEMA INTERLIGADO NACIONAL,44599.75,14141.78,7518.17,610.32,66870.02,66.7,21.15,11.24,0.91


### Participação média das fontes de geração

Com os dados mensais do SIN selecionados, é calculada a participação percentual média de cada fonte de geração ao longo do período de 2020 a 2025.

São consideradas as participações das fontes hidráulica, térmica, eólica e solar, sendo calculada a média dos 72 registros mensais e realizado o arredondamento para duas casas decimais.

O resultado fornece uma visão resumida da composição média das quatro fontes consideradas no projeto e será utilizado na visualização da distribuição da geração.

In [0]:
from pyspark.sql.functions import avg, round

df_resumo_participacao = (
    df_participacao_sin
    .agg(
        round(avg("participacao_hidraulica_pct"), 2).alias("hidraulica_pct"),
        round(avg("participacao_termica_pct"), 2).alias("termica_pct"),
        round(avg("participacao_eolica_pct"), 2).alias("eolica_pct"),
        round(avg("participacao_solar_pct"), 2).alias("solar_pct")
    )
)

display(df_resumo_participacao)

hidraulica_pct,termica_pct,eolica_pct,solar_pct
65.74,14.83,13.71,5.72


### Visualização da distribuição das fontes de geração

Para facilitar a interpretação da participação média das fontes de geração, os percentuais calculados anteriormente são organizados em uma estrutura contendo o nome de cada fonte e sua respectiva participação.

A partir desses dados é construído um gráfico de rosca, permitindo visualizar de forma comparativa a distribuição das fontes hidráulica, térmica, eólica e solar no SIN durante o período analisado.

A visualização evidencia a predominância da geração hidráulica entre as quatro fontes consideradas, enquanto as demais fontes apresentam participações médias menores no período de 2020 a 2025.

In [0]:
from pyspark.sql.functions import expr

df_grafico_participacao = (
    df_resumo_participacao
    .selectExpr(
        """
        stack(
            4,
            'Hidráulica', hidraulica_pct,
            'Térmica', termica_pct,
            'Eólica', eolica_pct,
            'Solar', solar_pct
        ) as (fonte, participacao_pct)
        """
    )
)

display(df_grafico_participacao)

fonte,participacao_pct
Hidráulica,65.74
Térmica,14.83
Eólica,13.71
Solar,5.72


Databricks visualization. Run in Databricks to view.

### Validação das participações percentuais

Como etapa de validação, é realizada a soma das participações percentuais médias das quatro fontes consideradas na análise.

O objetivo dessa verificação é confirmar a consistência dos percentuais utilizados na distribuição da geração. As participações das fontes hidráulica, térmica, eólica e solar são somadas e o resultado é arredondado para duas casas decimais.

A soma obtida foi de 100%, confirmando a consistência da distribuição percentual apresentada na análise.

In [0]:
from pyspark.sql.functions import col, round

df_validacao_participacao = (
    df_resumo_participacao
    .withColumn(
        "soma_percentuais",
        round(
            col("hidraulica_pct") +
            col("termica_pct") +
            col("eolica_pct") +
            col("solar_pct"),
            2
        )
    )
)

display(df_validacao_participacao)

hidraulica_pct,termica_pct,eolica_pct,solar_pct,soma_percentuais
65.74,14.83,13.71,5.72,100.0


### Análise da Pergunta 2

A análise da distribuição da geração no Sistema Interligado Nacional (SIN), considerando as quatro fontes presentes no conjunto de dados, demonstra predominância da geração hidráulica no período analisado.

Considerando a média das participações mensais entre 2020 e 2025, a distribuição observada foi:

- **Hidráulica:** 65,74%;
- **Térmica:** 14,83%;
- **Eólica:** 13,71%;
- **Solar:** 5,72%.

A geração hidráulica apresentou, portanto, a maior participação entre as quatro fontes consideradas. As fontes térmica e eólica apresentaram participações médias próximas entre si, enquanto a geração solar apresentou a menor participação média no período.

Em valores médios de geração, também foi observada a seguinte distribuição:

- **Hidráulica:** 47.370,91;
- **Térmica:** 10.554,89;
- **Eólica:** 9.988,17;
- **Solar:** 4.424,23.

Os percentuais apresentados correspondem à **média das participações mensais calculadas para as quatro fontes consideradas neste projeto**. Dessa forma, os resultados não devem ser interpretados como uma representação completa de toda a matriz elétrica brasileira.

Os resultados permitem concluir que, dentro do escopo analisado, a geração hidráulica apresentou participação predominante, enquanto as demais fontes contribuíram em proporções menores para a geração representada no conjunto de dados.


## Pergunta 3 — Evolução da participação das fontes de geração

**Como evoluiu a participação das diferentes fontes de geração de energia elétrica ao longo do período analisado?**

Para responder a essa pergunta, será analisada a evolução da participação das fontes hidráulica, térmica, eólica e solar entre 2020 e 2025.

A análise utilizará os registros correspondentes ao **Sistema Interligado Nacional (SIN)** disponíveis na camada Gold, permitindo acompanhar como a participação das quatro fontes consideradas no projeto se modificou ao longo do período.


### Preparação e visualização da evolução das fontes

Para analisar como a participação das diferentes fontes de geração evoluiu ao longo do tempo, são utilizados os registros mensais do SIN preparados anteriormente.

Os campos `ano` e `mes` são combinados para criar a coluna `data`, permitindo organizar cronologicamente as observações. Em seguida, são selecionadas as participações percentuais das fontes hidráulica, térmica, eólica e solar.

Os dados são ordenados cronologicamente e é verificada a quantidade de meses disponíveis. Para o período de 2020 a 2025, a análise contém 72 observações mensais.

A partir desses dados, é construído um gráfico de linhas que apresenta a evolução mensal da participação percentual das quatro fontes. Essa visualização permite identificar mudanças na composição da geração ao longo do período, bem como períodos de aumento ou redução da participação de cada fonte.

In [0]:
from pyspark.sql.functions import make_date, lit

df_evolucao_fontes = (
    df_participacao_sin
    .withColumn(
        "data",
        make_date("ano", "mes", lit(1))
    )
    .select(
        "data",
        "participacao_hidraulica_pct",
        "participacao_termica_pct",
        "participacao_eolica_pct",
        "participacao_solar_pct"
    )
    .orderBy("data")
)

print("Quantidade de meses:", df_evolucao_fontes.count())

display(df_evolucao_fontes)

Quantidade de meses: 72


data,participacao_hidraulica_pct,participacao_termica_pct,participacao_eolica_pct,participacao_solar_pct
2020-01-01,74.56,19.43,5.27,0.74
2020-02-01,79.9,13.69,5.66,0.75
2020-03-01,82.39,12.26,4.5,0.85
2020-04-01,78.33,13.23,7.38,1.06
2020-05-01,75.58,14.32,9.04,1.06
2020-06-01,72.68,14.57,11.69,1.07
2020-07-01,73.99,12.0,12.93,1.07
2020-08-01,71.2,12.86,14.87,1.08
2020-09-01,72.48,12.7,13.75,1.07
2020-10-01,66.7,21.15,11.24,0.91


Databricks visualization. Run in Databricks to view.

### Comparação da participação das fontes entre 2020 e 2025

Para identificar as principais mudanças ocorridas entre o início e o final do período analisado, são selecionados os registros do SIN correspondentes aos anos de 2020 e 2025.

Para cada ano, é calculada a participação percentual média das fontes hidráulica, térmica, eólica e solar, considerando os respectivos registros mensais.

Essa comparação permite observar como a composição das quatro fontes consideradas no projeto se modificou entre 2020 e 2025 e fornece a base para o cálculo da variação em pontos percentuais realizado na etapa seguinte.

In [0]:
from pyspark.sql.functions import avg, round

df_comparacao_fontes = (
    df_participacao_sin
    .filter(df_participacao_sin.ano.isin(2020, 2025))
    .groupBy("ano")
    .agg(
        round(avg("participacao_hidraulica_pct"), 2).alias("hidraulica_pct"),
        round(avg("participacao_termica_pct"), 2).alias("termica_pct"),
        round(avg("participacao_eolica_pct"), 2).alias("eolica_pct"),
        round(avg("participacao_solar_pct"), 2).alias("solar_pct")
    )
    .orderBy("ano")
)

display(df_comparacao_fontes)

ano,hidraulica_pct,termica_pct,eolica_pct,solar_pct
2020,73.06,16.13,9.86,0.95
2025,57.4,12.89,16.62,13.1


### Variação da participação das fontes entre 2020 e 2025

Para quantificar as mudanças observadas entre o início e o final do período, são organizadas as participações médias de cada fonte em 2020 e 2025 e calculada a diferença entre esses valores em pontos percentuais.

Valores positivos indicam aumento da participação da fonte, enquanto valores negativos indicam redução.

Essa comparação permite identificar de forma objetiva quais fontes ganharam ou perderam participação entre 2020 e 2025 dentro do conjunto das quatro fontes consideradas na análise.

In [0]:
from pyspark.sql.functions import col, round

dados_2020 = (
    df_comparacao_fontes
    .filter(col("ano") == 2020)
    .select(
        col("hidraulica_pct").alias("hidraulica_2020"),
        col("termica_pct").alias("termica_2020"),
        col("eolica_pct").alias("eolica_2020"),
        col("solar_pct").alias("solar_2020")
    )
)

dados_2025 = (
    df_comparacao_fontes
    .filter(col("ano") == 2025)
    .select(
        col("hidraulica_pct").alias("hidraulica_2025"),
        col("termica_pct").alias("termica_2025"),
        col("eolica_pct").alias("eolica_2025"),
        col("solar_pct").alias("solar_2025")
    )
)

comparacao = dados_2020.crossJoin(dados_2025)

df_variacao_fontes = comparacao.selectExpr(
    """
    stack(
        4,
        'Hidráulica', hidraulica_2020, hidraulica_2025,
        'Térmica', termica_2020, termica_2025,
        'Eólica', eolica_2020, eolica_2025,
        'Solar', solar_2020, solar_2025
    ) as (fonte, participacao_2020, participacao_2025)
    """
).withColumn(
    "variacao_pontos_percentuais",
    round(col("participacao_2025") - col("participacao_2020"), 2)
)

display(df_variacao_fontes)

fonte,participacao_2020,participacao_2025,variacao_pontos_percentuais
Hidráulica,73.06,57.4,-15.66
Térmica,16.13,12.89,-3.24
Eólica,9.86,16.62,6.76
Solar,0.95,13.1,12.15


### Validação das participações em 2020 e 2025

Como verificação de consistência da comparação realizada, são somadas as participações médias das fontes hidráulica, térmica, eólica e solar para os anos de 2020 e 2025.

O objetivo é confirmar que, mesmo com as mudanças observadas na participação de cada fonte, a composição percentual das quatro fontes consideradas permanece totalizando aproximadamente 100%.

Os resultados confirmam essa consistência nos dois anos. A pequena diferença numérica apresentada em 2025 decorre da precisão utilizada nos cálculos computacionais e não representa uma diferença relevante nos dados.

In [0]:
df_comparacao_fontes.select(
    "ano",
    (
        col("hidraulica_pct") +
        col("termica_pct") +
        col("eolica_pct") +
        col("solar_pct")
    ).alias("soma_participacoes")
).display()

ano,soma_participacoes
2020,100.0
2025,100.00999999999999


### Análise da Pergunta 3

A evolução mensal das participações das fontes de geração entre 2020 e 2025 demonstra mudanças na composição das quatro fontes consideradas no Sistema Interligado Nacional (SIN).

A geração **hidráulica permaneceu como a fonte de maior participação** durante o período analisado, porém sua participação média anual passou de **73,06% em 2020 para 57,40% em 2025**, representando uma redução de **15,66 pontos percentuais**.

A geração **térmica** também apresentou redução na comparação entre os dois anos, passando de **16,13% em 2020 para 12,89% em 2025**, uma diferença de **-3,24 pontos percentuais**.

Em sentido contrário, as fontes **eólica e solar aumentaram sua participação**. A geração eólica passou de **9,86% para 16,62%**, correspondendo a um aumento de **6,76 pontos percentuais**. A geração solar apresentou a maior ampliação entre as fontes analisadas, passando de **0,95% em 2020 para 13,10% em 2025**, um aumento de **12,15 pontos percentuais**.

O gráfico mensal também demonstra que essas participações apresentam oscilações ao longo do período, indicando que a composição da geração não permanece constante ao longo dos meses.

Dessa forma, dentro do escopo das quatro fontes analisadas, os dados evidenciam uma mudança na composição da geração entre 2020 e 2025, caracterizada pela redução da participação hidráulica e térmica e pelo aumento da participação eólica e solar.

A análise é descritiva e não permite determinar, isoladamente, as causas dessas mudanças, pois fatores externos capazes de explicá-las não fazem parte do conjunto de dados utilizado neste MVP.


## Pergunta 4 — Relação entre geração e carga

**Como o comportamento da geração de energia elétrica se relaciona com as variações da carga nos diferentes subsistemas do SIN?**

Para responder a essa pergunta, serão analisados conjuntamente os valores médios mensais de carga e de geração das quatro fontes consideradas no projeto.

A análise será realizada por subsistema ao longo do período de 2020 a 2025. Também será considerado o intercâmbio líquido médio disponível no conjunto de dados, auxiliando na interpretação das diferenças entre a geração representada e a carga de cada subsistema.

A comparação tem caráter descritivo e busca identificar padrões de comportamento e associação entre as variáveis, sem estabelecer relações de causalidade.

### Carregamento dos dados de geração e carga

Para analisar a relação entre o comportamento da geração e as variações da carga, é carregada a tabela `geracao_carga_mensal` da camada Gold.

Essa tabela reúne, na mesma granularidade mensal por identificador de subsistema, a carga média, os valores médios das quatro fontes de geração consideradas no projeto, a geração total média calculada a partir dessas fontes e o intercâmbio médio de energia.

Após o carregamento, são verificadas a quantidade de registros e de colunas e exibida uma amostra dos dados. A tabela possui 360 registros e 11 atributos, correspondentes aos dados mensais do período de 2020 a 2025 para os cinco identificadores presentes no conjunto de dados.

Essa estrutura será utilizada nas próximas etapas para comparar geração e carga e avaliar a associação entre essas variáveis nos diferentes subsistemas.

In [0]:
tabela_gold_geracao_carga = "workspace.energia_gold.geracao_carga_mensal"

df_geracao_carga = spark.table(tabela_gold_geracao_carga)

print("Quantidade de registros:", df_geracao_carga.count())
print("Quantidade de colunas:", len(df_geracao_carga.columns))

display(df_geracao_carga.limit(20))

Quantidade de registros: 360
Quantidade de colunas: 11


ano,mes,id_subsistema,nom_subsistema,carga_media,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media,intercambio_liquido_medio
2020,1,N,NORTE,5412.0,8188.34,2861.9,126.65,0.0,11176.89,5764.89
2020,1,NE,NORDESTE,10952.56,2897.49,2322.59,2965.7,317.59,8503.37,-2449.19
2020,1,S,SUL,12599.13,4523.55,1817.24,517.36,0.0,6858.15,-5740.99
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,35469.7,6309.17,0.0,192.27,41971.14,2393.58
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,51079.08,13310.89,3609.71,509.87,68509.55,-31.72
2020,2,N,NORTE,5513.53,12613.98,2140.81,99.05,0.0,14853.84,9340.31
2020,2,NE,NORDESTE,10995.71,2883.37,1697.99,3086.87,333.01,8001.24,-2994.47
2020,2,S,SUL,12781.99,3361.24,1087.89,708.2,0.0,5157.33,-7624.67
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,36109.29,4494.53,0.0,180.45,40784.27,1278.81
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,54967.88,9421.22,3894.12,513.46,68796.68,-0.01


### Seleção dos quatro subsistemas regionais

Para realizar a análise individual dos subsistemas, o registro agregado do SIN é retirado desta etapa, mantendo apenas Norte, Nordeste, Sul e Sudeste/Centro-Oeste.

Os registros são organizados por ano, mês e identificador do subsistema, preservando os indicadores mensais de carga, geração e intercâmbio preparados na camada Gold.

Ao final, é verificada a quantidade de registros disponíveis. Foram obtidas 288 observações, correspondentes a 72 meses para cada um dos quatro subsistemas regionais.

Essa separação permite avaliar o comportamento da geração e da carga em cada subsistema sem misturar os dados regionais com o nível agregado do SIN.

In [0]:
df_geracao_carga_subsistemas = (
    df_geracao_carga
    .filter("id_subsistema != 'SIN'")
    .orderBy("ano", "mes", "id_subsistema")
)

print(
    "Quantidade de registros dos quatro subsistemas:",
    df_geracao_carga_subsistemas.count()
)

display(df_geracao_carga_subsistemas.limit(20))

Quantidade de registros dos quatro subsistemas: 288


ano,mes,id_subsistema,nom_subsistema,carga_media,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media,intercambio_liquido_medio
2020,1,N,NORTE,5412.0,8188.34,2861.9,126.65,0.0,11176.89,5764.89
2020,1,NE,NORDESTE,10952.56,2897.49,2322.59,2965.7,317.59,8503.37,-2449.19
2020,1,S,SUL,12599.13,4523.55,1817.24,517.36,0.0,6858.15,-5740.99
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,35469.7,6309.17,0.0,192.27,41971.14,2393.58
2020,2,N,NORTE,5513.53,12613.98,2140.81,99.05,0.0,14853.84,9340.31
2020,2,NE,NORDESTE,10995.71,2883.37,1697.99,3086.87,333.01,8001.24,-2994.47
2020,2,S,SUL,12781.99,3361.24,1087.89,708.2,0.0,5157.33,-7624.67
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,36109.29,4494.53,0.0,180.45,40784.27,1278.81
2020,3,N,NORTE,5347.47,15223.41,934.11,52.42,0.0,16209.94,10862.46
2020,3,NE,NORDESTE,10597.1,2962.22,1317.75,2278.28,366.93,6925.18,-3671.91


### Correlação entre carga e geração por subsistema

Para avaliar a associação entre o comportamento da carga e da geração ao longo do período analisado, é calculado o coeficiente de correlação entre a `carga_media` e a `geracao_total_media` para cada um dos quatro subsistemas regionais.

O coeficiente de correlação permite observar se as duas variáveis apresentam tendência de variação conjunta. Valores positivos indicam que carga e geração tendem a variar na mesma direção, enquanto valores negativos indicam tendência de variação em direções opostas. Valores mais próximos de zero representam uma associação linear mais fraca.

O cálculo é realizado separadamente para cada subsistema e os resultados são arredondados para três casas decimais. Essa análise possui caráter descritivo e não estabelece relação de causalidade entre carga e geração.

In [0]:
from pyspark.sql.functions import corr, round

df_correlacao = (
    df_geracao_carga_subsistemas
    .groupBy(
        "id_subsistema",
        "nom_subsistema"
    )
    .agg(
        round(
            corr("carga_media", "geracao_total_media"),
            3
        ).alias("correlacao_carga_geracao")
    )
    .orderBy("id_subsistema")
)

display(df_correlacao)

id_subsistema,nom_subsistema,correlacao_carga_geracao
N,NORTE,-0.294
NE,NORDESTE,0.65
S,SUL,0.329
SE,SUDESTE/CENTRO-OESTE,0.743


### Análise do intercâmbio de energia entre os subsistemas

Para complementar a análise da relação entre geração e carga, este bloco resume o comportamento do intercâmbio líquido de energia nos quatro subsistemas regionais.

Os dados são agrupados por subsistema e são calculados o valor médio, o valor mínimo e o valor máximo do intercâmbio ao longo do período analisado.

Essa informação auxilia na interpretação das diferenças entre a geração representada e a carga de cada subsistema, considerando que os subsistemas podem apresentar fluxos de intercâmbio de energia ao longo do tempo.

In [0]:
from pyspark.sql.functions import avg, min, max, round

df_resumo_intercambio = (
    df_geracao_carga_subsistemas
    .groupBy(
        "id_subsistema",
        "nom_subsistema"
    )
    .agg(
        round(avg("intercambio_liquido_medio"), 2).alias("intercambio_medio"),
        round(min("intercambio_liquido_medio"), 2).alias("intercambio_minimo"),
        round(max("intercambio_liquido_medio"), 2).alias("intercambio_maximo")
    )
    .orderBy("id_subsistema")
)

display(df_resumo_intercambio)

id_subsistema,nom_subsistema,intercambio_medio,intercambio_minimo,intercambio_maximo
N,NORTE,3455.05,-2300.48,12053.05
NE,NORDESTE,3873.31,-3671.91,9946.12
S,SUL,-2188.35,-8138.03,4171.73
SE,SUDESTE/CENTRO-OESTE,-4987.02,-11456.93,2393.58


### Preparação dos dados para comparação entre geração, carga e intercâmbio

Este bloco organiza os dados mensais dos quatro subsistemas regionais para realizar uma análise conjunta entre **carga média**, **geração total média** e **intercâmbio líquido médio**.

Também é criada uma coluna de data a partir do ano e do mês, permitindo acompanhar a evolução dessas variáveis ao longo do período de 2020 a 2025.

Ao final, são obtidos **288 registros**, correspondentes aos 72 meses analisados para cada um dos quatro subsistemas regionais. Essa estrutura será utilizada nas próximas etapas para comparar visualmente o comportamento da geração e da carga e avaliar a participação do intercâmbio nessa relação.

In [0]:
from pyspark.sql.functions import make_date, lit

df_relacao_geracao_carga = (
    df_geracao_carga_subsistemas
    .withColumn(
        "data",
        make_date("ano", "mes", lit(1))
    )
    .select(
        "data",
        "id_subsistema",
        "nom_subsistema",
        "carga_media",
        "geracao_total_media",
        "intercambio_liquido_medio"
    )
    .orderBy("data", "id_subsistema")
)

print("Quantidade de registros:", df_relacao_geracao_carga.count())

display(df_relacao_geracao_carga)

Quantidade de registros: 288


data,id_subsistema,nom_subsistema,carga_media,geracao_total_media,intercambio_liquido_medio
2020-01-01,N,NORTE,5412.0,11176.89,5764.89
2020-01-01,NE,NORDESTE,10952.56,8503.37,-2449.19
2020-01-01,S,SUL,12599.13,6858.15,-5740.99
2020-01-01,SE,SUDESTE/CENTRO-OESTE,39577.57,41971.14,2393.58
2020-02-01,N,NORTE,5513.53,14853.84,9340.31
2020-02-01,NE,NORDESTE,10995.71,8001.24,-2994.47
2020-02-01,S,SUL,12781.99,5157.33,-7624.67
2020-02-01,SE,SUDESTE/CENTRO-OESTE,39505.45,40784.27,1278.81
2020-03-01,N,NORTE,5347.47,16209.94,10862.46
2020-03-01,NE,NORDESTE,10597.1,6925.18,-3671.91


### Comparação entre carga e geração no subsistema Norte

Este bloco seleciona exclusivamente os registros do **subsistema Norte** e organiza os valores mensais de **carga média** e **geração total média** entre 2020 e 2025.

Os dados são ordenados cronologicamente e apresentados em um gráfico de linhas com **72 meses**, permitindo comparar visualmente a evolução das duas variáveis ao longo do período.

Essa visualização auxilia na identificação de períodos em que geração e carga apresentam comportamentos semelhantes ou distintos. A comparação possui caráter descritivo e não representa, isoladamente, uma relação de causa e efeito.

In [0]:
df_relacao_norte = (
    df_relacao_geracao_carga
    .filter("id_subsistema = 'N'")
    .select(
        "data",
        "carga_media",
        "geracao_total_media"
    )
    .orderBy("data")
)

print("Quantidade de meses:", df_relacao_norte.count())

display(df_relacao_norte)

Quantidade de meses: 72


data,carga_media,geracao_total_media
2020-01-01,5412.0,11176.89
2020-02-01,5513.53,14853.84
2020-03-01,5347.47,16209.94
2020-04-01,4921.12,11983.56
2020-05-01,5111.48,11804.6
2020-06-01,5286.12,11174.13
2020-07-01,5387.09,6429.33
2020-08-01,5753.43,5923.72
2020-09-01,5918.47,6178.59
2020-10-01,5961.54,7300.68


Databricks visualization. Run in Databricks to view.

### Resumo da carga e da geração por subsistema

Este bloco calcula uma visão consolidada do período de **2020 a 2025** para cada um dos quatro subsistemas regionais do SIN.

Para cada subsistema, são calculadas a **carga média do período** e a **geração total média do período**, permitindo comparar o comportamento geral dessas duas variáveis.

Os resultados mostram diferenças entre geração e carga nos subsistemas, o que reforça a importância de considerar também o intercâmbio líquido disponível no conjunto de dados na interpretação dessas diferenças. Essa relação será verificada nos próximos blocos da análise.

In [0]:
from pyspark.sql.functions import avg, round

df_resumo_geracao_carga = (
    df_geracao_carga_subsistemas
    .groupBy("id_subsistema", "nom_subsistema")
    .agg(
        round(avg("carga_media"), 2).alias("carga_media_periodo"),
        round(avg("geracao_total_media"), 2).alias("geracao_media_periodo")
    )
    .orderBy("id_subsistema")
)

display(df_resumo_geracao_carga)

id_subsistema,nom_subsistema,carga_media_periodo,geracao_media_periodo
N,NORTE,6829.59,10284.65
NE,NORDESTE,11896.91,15770.22
S,SUL,12491.5,10302.53
SE,SUDESTE/CENTRO-OESTE,40967.82,35980.81


### Diferença entre geração e carga por subsistema

Este bloco calcula a diferença entre a **geração total média** e a **carga média** de cada subsistema no período de 2020 a 2025.

Os resultados mostram diferenças positivas para os subsistemas **Norte** e **Nordeste**, enquanto **Sul** e **Sudeste/Centro-Oeste** apresentam diferenças negativas.

Essa comparação permite identificar que geração e carga não apresentam os mesmos valores em cada subsistema. Na próxima etapa, essas diferenças serão comparadas com o **intercâmbio líquido médio**, permitindo verificar como o intercâmbio entre os subsistemas se relaciona com os valores observados.

In [0]:
from pyspark.sql.functions import col, round

df_resumo_diferenca = (
    df_resumo_geracao_carga
    .withColumn(
        "diferenca_geracao_carga",
        round(
            col("geracao_media_periodo") - col("carga_media_periodo"),
            2
        )
    )
)

display(df_resumo_diferenca)

id_subsistema,nom_subsistema,carga_media_periodo,geracao_media_periodo,diferenca_geracao_carga
N,NORTE,6829.59,10284.65,3455.06
NE,NORDESTE,11896.91,15770.22,3873.31
S,SUL,12491.5,10302.53,-2188.97
SE,SUDESTE/CENTRO-OESTE,40967.82,35980.81,-4987.01


### Validação da diferença entre geração, carga e intercâmbio

Este bloco compara a **diferença entre a geração total média e a carga média** de cada subsistema com o respectivo **intercâmbio líquido médio**.

Os resultados apresentam valores muito próximos entre essas duas medidas. As diferenças residuais são pequenas e decorrem principalmente do arredondamento aplicado durante as etapas de agregação dos dados.

Essa validação evidencia a importância de considerar o intercâmbio líquido na interpretação das diferenças observadas entre a geração representada e a carga de cada subsistema. Dessa forma, a análise regional é complementada pela informação de intercâmbio disponível no conjunto de dados, sem atribuir neste MVP um sentido específico aos sinais positivos ou negativos sem a confirmação da convenção adotada pela fonte.

In [0]:
df_validacao_balanco = (
    df_resumo_diferenca.alias("d")
    .join(
        df_resumo_intercambio.alias("i"),
        on=["id_subsistema", "nom_subsistema"],
        how="inner"
    )
    .select(
        "id_subsistema",
        "nom_subsistema",
        "diferenca_geracao_carga",
        "intercambio_medio",
        round(
            col("diferenca_geracao_carga") - col("intercambio_medio"),
            2
        ).alias("diferenca_validacao")
    )
    .orderBy("id_subsistema")
)

display(df_validacao_balanco)

id_subsistema,nom_subsistema,diferenca_geracao_carga,intercambio_medio,diferenca_validacao
N,NORTE,3455.06,3455.05,0.01
NE,NORDESTE,3873.31,3873.31,0.0
S,SUL,-2188.97,-2188.35,-0.62
SE,SUDESTE/CENTRO-OESTE,-4987.01,-4987.02,0.01


### Correlação entre geração e carga no Sistema Interligado Nacional

Este bloco calcula a **correlação entre a carga média e a geração total média** considerando os registros agregados do Sistema Interligado Nacional (SIN) ao longo do período analisado.

O coeficiente de correlação obtido foi de **0,994**, valor muito próximo de 1. Esse resultado indica uma **forte associação positiva** entre as duas variáveis: ao longo do período, os aumentos e reduções da carga tendem a ser acompanhados por movimentos semelhantes na geração total representada no conjunto de dados.

Esse resultado complementa a análise realizada por subsistema e evidencia que, quando o SIN é observado de forma agregada, geração e carga apresentam comportamento fortemente relacionado. A correlação, entretanto, representa uma associação estatística e **não deve ser interpretada como evidência de causalidade**.

In [0]:
from pyspark.sql.functions import corr, round

df_correlacao_sin = (
    df_geracao_carga
    .filter("id_subsistema = 'SIN'")
    .agg(
        round(
            corr("carga_media", "geracao_total_media"),
            3
        ).alias("correlacao_carga_geracao_sin")
    )
)

display(df_correlacao_sin)

correlacao_carga_geracao_sin
0.994


### Análise da Pergunta 4

A análise conjunta da carga e da geração demonstra que a relação entre essas variáveis apresenta comportamentos diferentes entre os subsistemas do Sistema Interligado Nacional.

Considerando os valores médios mensais entre 2020 e 2025, a correlação entre a carga média e a geração total média das quatro fontes analisadas foi de:

- **Norte:** -0,294;
- **Nordeste:** 0,650;
- **Sul:** 0,329;
- **Sudeste/Centro-Oeste:** 0,743.

Os resultados mostram que a associação entre carga e geração não ocorre com a mesma intensidade em todos os subsistemas. Entre os valores observados, Sudeste/Centro-Oeste e Nordeste apresentaram as maiores correlações positivas, enquanto o Sul apresentou correlação positiva de menor magnitude e o Norte apresentou correlação negativa no período analisado.

Quando o Sistema Interligado Nacional é analisado de forma agregada, a correlação entre carga e geração foi de **0,994**, indicando uma associação linear positiva muito elevada entre as duas variáveis no conjunto do sistema.

A análise também mostrou que a diferença entre a geração média das quatro fontes consideradas e a carga média de cada subsistema apresenta valores muito próximos do intercâmbio líquido médio registrado nos dados. As diferenças encontradas nessa validação foram de **0,01 no Norte, 0,00 no Nordeste, -0,62 no Sul e 0,01 no Sudeste/Centro-Oeste**.

Esse resultado evidencia a importância de considerar o intercâmbio na interpretação do comportamento regional da geração e da carga. Entretanto, neste MVP não é atribuída uma interpretação de entrada ou saída de energia aos sinais positivos e negativos do intercâmbio sem a confirmação da convenção adotada pela fonte dos dados.

Dessa forma, os dados indicam que a relação entre geração e carga possui características distintas nos subsistemas, enquanto no SIN agregado as duas variáveis apresentam comportamento fortemente associado ao longo do período analisado.

Os resultados apresentados são descritivos. As correlações identificadas representam associação estatística entre as variáveis e não estabelecem relação de causalidade.